# Data quality checking for Eniac project

In [3]:
import pandas as pd
pd.set_option("display.float_format", lambda x: "%.2f" % x)
pd.set_option("display.max_rows", 20)

In [8]:
products = pd.read_csv('C:/Users/admin/Documents/1WBS/3Data_Cln_Stry/eniac_cleaning/data/interim/products_cl.csv')
orders = pd.read_csv('C:/Users/admin/Documents/1WBS/3Data_Cln_Stry/eniac_cleaning/data/interim/orders_cl.csv')
orderlines = pd.read_csv('C:/Users/admin/Documents/1WBS/3Data_Cln_Stry/eniac_cleaning/data/interim/orderlines_cl.csv')


### Check the consistency of keys between tables

In [ ]:
#Set intersection of orders and orderlines
common_ids = set(orders.order_id) & set(orderlines.order_id)
orders_qu = orders[orders.order_id.isin(common_ids)]
orderlines_qu = orderlines[orderlines.order_id.isin(common_ids)]

8752 rows in orderlines with no row in products
2484 rows in products with no row in orderlines


In [21]:
#Set intersection of products and orderlines
common_ids = set(orderlines_qu.sku) & set(products.sku)

#size of the problem
print(str(orderlines[~orderlines.sku.isin(common_ids)].shape[0]) + " rows in orderlines with no row in products")
print(str(products[~products.sku.isin(common_ids)].shape[0]) + " rows in products with no row in orderlines")

#Find the orders with a missing product by finding the rows in the orderline table and then the set of orders associated with this
affected_orders = set(orderlines[~orderlines.sku.isin(common_ids)].order_id)

#remove these orders from the orders and orderlines tables
orders_qu = orders_qu[~orders_qu.order_id.isin(affected_orders)]
orderlines_qu = orderlines_qu[~orderlines_qu.order_id.isin(affected_orders)]

#Check consistency
print(orders_qu.order_id.nunique())
print(orderlines_qu.order_id.nunique())

8805 rows in orderlines with no row in products
2522 rows in products with no row in orderlines
196211
196211


### Check numerical conistency for orders

In [42]:
#Add new column for revenue: quantity * price
orderlines_qu['revenue'] = orderlines_qu.unit_price * orderlines_qu.product_quantity
#Group orderlines by order_id to find total paid per order
orderlines_grped = orderlines_qu.groupby('order_id', as_index=False)['revenue'].sum()
#merge this back into the orders table to compare against total_paid from this column
compare_df = orders_qu.merge(orderlines_grped, on='order_id')
#calculate the difference between the two measures
compare_df['diff'] = compare_df.total_paid - compare_df.revenue

#Flag any differences outside of a certain tolerance
compare_df['suspicious'] = ~(compare_df['diff'].between(-1, 20))
#Find all order ids with suspicious orders
suspicious_ids = set(compare_df[compare_df.suspicious].order_id)

#produce final tables
orders_final = orders_qu[~orders_qu.order_id.isin(suspicious_ids)]
orderlines_final = orderlines_qu[~orderlines_qu.order_id.isin(suspicious_ids)]

#Check consistency
print(orders_final.order_id.nunique())
print(orderlines_final.order_id.nunique())


195718
195718


In [ ]:
compare_df[compare_df.suspicious]
#plot differences
compare_df.loc[compare_df["diff"].between(-50, 50), "diff"].hist(bins=60, figsize=(12, 4));


,order_id,created_date,total_paid,state,revenue,diff,suspicious
8,246405,2017-11-24 10:01:27,407.96,Completed,275.75,132.21,True
21,254537,2017-05-23 19:58:30,102.97,Pending,32.99,69.98,True
47,264244,2018-01-29 15:33:06,141.97,Completed,69.99,71.98,True
61,269083,2017-12-01 23:52:02,139.27,Completed,84.28,54.99,True
69,274043,2017-01-02 16:33:47,3.98,Completed,24.99,-21.01,True
...,...,...,...,...,...,...,...
186029,515325,2018-02-21 11:31:20,13.19,Shopping Basket,3038.00,-3024.81,True
186439,515818,2018-02-22 01:02:35,6.59,Shopping Basket,388.06,-381.47,True
189759,519801,2018-03-02 08:59:34,31.22,Shopping Basket,7771.23,-7740.01,True
192945,523652,2018-03-09 10:40:47,167.05,Completed,147.04,20.01,True
